In [16]:
# imports
import numpy as np
import matplotlib.pyplot as plt

In [17]:
# set which one-hot sample to compare (0-4)
# 0: corner (0,0)
# 1: corner (47,47)
# 2: center (24,24)
# 3: top edge (0,24)
# 4: left edge (24,0)
TEST_NUM = 1

In [18]:
def load_rtl_layer(path, nfrac):
    raw = np.loadtxt(path, delimiter=',')   # shape (n_positions, n_channels)
    flat = raw.flatten()                     # row-major: channel varies fastest, matches (H,W,C) flatten
    return flat / (2 ** nfrac)

def load_ref_layer(path):
    return np.loadtxt(path, delimiter=',')   # already flat, already decimal

# every layer in pipeline order:
#   (short name, rtl filename suffix, hls4ml filename suffix, nfrac)
LAYER_SPECS = [
    ("conv0",  "conv0",  "q_activation",    6),
    ("pool0",  "pool0",  "max_pooling2d",   6),
    ("conv1",  "conv1",  "q_activation_1",  5),
    ("pool1",  "pool1",  "max_pooling2d_1", 5),
    ("conv2",  "conv2",  "q_activation_2",  5),
    ("pool2",  "pool2",  "max_pooling2d_2", 5),
    ("dense0", "dense0", "q_dense",         5),
    ("relu0",  "relu0",  "q_activation_3",  5),
    ("dense1", "dense1", "q_dense_1",       5),
    ("relu1",  "relu1",  "q_activation_4",  5),
    ("final",  "final",  "q_dense_2",       5),
]

def rtl_path(test_num, suffix):
    return f"round_up_traces/rtl_{test_num}_{suffix}.csv"

def hls_path(test_num, suffix):
    return f"traces/hls4ml_{test_num}_{suffix}.csv"

def keras_path(test_num, suffix):
    return f"traces/keras_{test_num}_{suffix}.csv"

In [19]:
# quick single-layer sanity check (conv0) for the selected TEST_NUM
rtl_conv0 = load_rtl_layer(rtl_path(TEST_NUM, "conv0"), nfrac=6)
hls4ml_conv0 = load_ref_layer(hls_path(TEST_NUM, "q_activation"))

print(rtl_conv0.shape, hls4ml_conv0.shape)    # RTL may be a few positions short
                                              # if finalOutputValid fired (and the testbench
                                              # closed the trace files) before conv0 finished
                                              # streaming its full raster scan -- expected,
                                              # not a bug.

n = min(len(rtl_conv0), len(hls4ml_conv0))
diff = np.abs(rtl_conv0[:n] - hls4ml_conv0[:n])
print("mean:", diff.mean())
print("median:", np.median(diff))
print("p95:", np.percentile(diff, 95))
print("max:", diff.max())
print(f"num outliers (>0.1): {(diff > 0.1).sum()} / {diff.size}")

(12684,) (12696,)
mean: 0.0
median: 0.0
p95: 0.0
max: 0.0
num outliers (>0.1): 0 / 12684


In [20]:
def diff_stats(rtl_p, ref_p, nfrac, label=None, outlier_threshold=0.1):
    rtl = load_rtl_layer(rtl_p, nfrac)
    ref = load_ref_layer(ref_p)
    n = min(len(rtl), len(ref))
    note = ""
    if len(rtl) != len(ref):
        note = f"  [trimmed: rtl={len(rtl)}, ref={len(ref)} -> {n}]"
    diff = np.abs(rtl[:n] - ref[:n])
    stats = {
        "label": label or ref_p,
        "mean": diff.mean(),
        "median": np.median(diff),
        "p95": np.percentile(diff, 95),
        "max": diff.max(),
        "outliers": int((diff > outlier_threshold).sum()),
        "outlier_threshold": outlier_threshold,
        "n": diff.size,
    }
    print(f"{stats['label']:12s} mean={stats['mean']:.6f} median={stats['median']:.6f} "
          f"p95={stats['p95']:.6f} max={stats['max']:.6f} outliers={stats['outliers']}/{stats['n']} (>{outlier_threshold}){note}")
    return stats

def run_full_comparison(test_num, ref_source="hls4ml"):
    """
    Runs diff_stats for every layer in LAYER_SPECS, for the given TEST_NUM.
    ref_source: 'hls4ml' or 'keras' -- which reference trace to compare RTL against.
    Returns a list of per-layer stat dicts (in pipeline order).
    """
    print(f"=== TEST_NUM={test_num}  (ref={ref_source}) ===")
    results = []
    for name, rtl_suffix, hls_suffix, nfrac in LAYER_SPECS:
        rp = rtl_path(test_num, rtl_suffix)
        refp = hls_path(test_num, hls_suffix) if ref_source == "hls4ml" else keras_path(test_num, hls_suffix)
        results.append(diff_stats(rp, refp, nfrac, label=name))
    return results

results = run_full_comparison(TEST_NUM, ref_source="hls4ml")

=== TEST_NUM=1  (ref=hls4ml) ===
conv0        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/12684 (>0.1)  [trimmed: rtl=12684, ref=12696 -> 12684]
pool0        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/726 (>0.1)
conv1        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/648 (>0.1)
pool1        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/128 (>0.1)
conv2        mean=0.003125 median=0.000000 p95=0.031250 max=0.031250 outliers=0/40 (>0.1)
pool2        mean=0.003125 median=0.000000 p95=0.017187 max=0.031250 outliers=0/10 (>0.1)
dense0       mean=0.010417 median=0.000000 p95=0.031250 max=0.031250 outliers=0/15 (>0.1)
relu0        mean=0.002083 median=0.000000 p95=0.009375 max=0.031250 outliers=0/15 (>0.1)
dense1       mean=0.021875 median=0.031250 p95=0.048437 max=0.062500 outliers=0/10 (>0.1)
relu1        mean=0.003125 median=0.000000 p95=0.017187 max=0.031250 outliers=0/10 (>0.1)
final        mean=0

In [21]:
# summarize just the final-layer error per test.
def summarize_final_across_tests(test_nums=(0, 1, 2, 3, 4), ref_source="hls4ml"):
    print(f"{'test':>4s}  {'mean':>10s}  {'median':>10s}  {'p95':>10s}  {'max':>10s}  outliers")
    for t in test_nums:
        try:
            s = diff_stats(
                rtl_path(t, "final"),
                hls_path(t, "q_dense_2") if ref_source == "hls4ml" else keras_path(t, "q_dense_2"),
                nfrac=5,
                label=f"test {t}",
            )
        except OSError as e:
            print(f"test {t}: no data yet ({e})")

summarize_final_across_tests()

test        mean      median         p95         max  outliers
test 0       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5 (>0.1)
test 1       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5 (>0.1)
test 2       mean=0.068750 median=0.031250 p95=0.125000 max=0.125000 outliers=2/5 (>0.1)
test 3       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5 (>0.1)
test 4       mean=0.050000 median=0.031250 p95=0.093750 max=0.093750 outliers=0/5 (>0.1)


In [22]:
# outlier inspector for a specific layer -- shows which raw indices/channels
# exceed the outlier threshold, and their actual RTL vs. reference values.
def show_outliers(test_num, rtl_suffix, hls_suffix, nfrac, n_channels, threshold=0.1, ref_source="hls4ml"):
    rtl = load_rtl_layer(rtl_path(test_num, rtl_suffix), nfrac)
    refp = hls_path(test_num, hls_suffix) if ref_source == "hls4ml" else keras_path(test_num, hls_suffix)
    ref = load_ref_layer(refp)
    n = min(len(rtl), len(ref))
    diff = np.abs(rtl[:n] - ref[:n])
    outlier_idx = np.where(diff > threshold)[0]
    print(f"outlier indices: {outlier_idx} -> channels: {outlier_idx % n_channels}")
    print("rtl:   ", rtl[outlier_idx])
    print(f"{ref_source}:", ref[outlier_idx])
    return outlier_idx

In [31]:
print("=== RTL vs Keras, TEST_NUM=2 ===")
results_2_keras = run_full_comparison(2, ref_source="keras")

print("=== RTL vs hls4ml, TEST_NUM=2 (for comparison) ===")
results_2_hls = run_full_comparison(2, ref_source="hls4ml")

=== RTL vs Keras, TEST_NUM=2 ===
=== TEST_NUM=2  (ref=keras) ===
conv0        mean=0.000007 median=0.000000 p95=0.000000 max=0.015625 outliers=0/12684 (>0.1)  [trimmed: rtl=12684, ref=12696 -> 12684]
pool0        mean=0.000108 median=0.000000 p95=0.000000 max=0.015625 outliers=0/726 (>0.1)
conv1        mean=0.018953 median=0.000000 p95=0.103906 max=0.843750 outliers=33/648 (>0.1)
pool1        mean=0.052490 median=0.000000 p95=0.317187 max=0.843750 outliers=21/128 (>0.1)
conv2        mean=0.071094 median=0.000000 p95=0.346875 max=0.468750 outliers=9/40 (>0.1)
pool2        mean=0.098437 median=0.046875 p95=0.335156 max=0.468750 outliers=3/10 (>0.1)
dense0       mean=0.292253 median=0.198730 p95=0.639404 max=0.815430 outliers=13/15 (>0.1)
relu0        mean=0.058333 median=0.000000 p95=0.203125 max=0.312500 outliers=5/15 (>0.1)
dense1       mean=0.167969 median=0.104736 p95=0.470215 max=0.540527 outliers=5/10 (>0.1)
relu1        mean=0.034375 median=0.000000 p95=0.181250 max=0.265625 outli

In [23]:
print("=== TEST_NUM=2 (center) ===")
_ = show_outliers(2, "final", "q_dense_2", nfrac=5, n_channels=5, threshold=0.0)  # threshold=0 shows ALL 5 values, not just >0.1

print("=== TEST_NUM=4 (left edge) ===")
_ = show_outliers(4, "final", "q_dense_2", nfrac=5, n_channels=5, threshold=0.0)

=== TEST_NUM=2 (center) ===
outlier indices: [0 1 2 3 4] -> channels: [0 1 2 3 4]
rtl:    [0.46875 0.46875 0.3125  0.3125  0.96875]
hls4ml: [0.5    0.5    0.1875 0.1875 1.    ]
=== TEST_NUM=4 (left edge) ===
outlier indices: [0 1 2 3] -> channels: [0 1 2 3]
rtl:    [0.4375  0.46875 0.25    0.25   ]
hls4ml: [0.46875 0.5     0.15625 0.15625]


In [47]:
rtl = load_rtl_layer(rtl_path(2, "conv1"), nfrac=5)
ref = load_ref_layer(hls_path(2, "q_activation_1"))
n = min(len(rtl), len(ref))

# isolate just channel 5 across every spatial position
# ch5_rtl = rtl[5:n:8]
# ch5_ref = ref[5:n:8]

# print("rtl channel 5 (first 20):", ch5_rtl[:20])
# print("ref channel 5 (first 20):", ch5_ref[:20])
# print("rtl channel 5 unique values:", np.unique(ch5_rtl))

diff = np.abs(rtl[:n] - ref[:n])

# sort outliers by size, biggest first
outlier_idx = np.where(diff > 0.1)[0]
sorted_idx = outlier_idx[np.argsort(-diff[outlier_idx])]

for idx in sorted_idx[:10]:
    print(f"idx={idx} (channel {idx % 8}, position {idx // 8}): rtl={rtl[idx]:.5f}, hls4ml={ref[idx]:.5f}, diff={diff[idx]:.5f}")

outlier_channels = outlier_idx % 8
unique, counts = np.unique(outlier_channels, return_counts=True)
for ch, cnt in zip(unique, counts):
    ch_diffs = diff[outlier_idx][outlier_channels == ch]
    print(f"channel {ch}: {cnt} outliers, max diff={ch_diffs.max():.5f}, mean diff={ch_diffs.mean():.5f}")

ref_pre = load_ref_layer(hls_path(2, "q_conv2d_batchnorm_1"))  # hls4ml's OWN pre-ReLU value, for diagnosis only

for idx in [331, 249, 409, 265, 325, 323, 326, 395, 413, 396]:
    print(f"idx={idx} (ch {idx%8}): rtl_post_relu={rtl[idx]:.5f}, hls4ml_pre_relu={ref_pre[idx]:.5f}, hls4ml_post_relu={ref[idx]:.5f}")


idx=331 (channel 3, position 41): rtl=0.84375, hls4ml=0.00000, diff=0.84375
idx=249 (channel 1, position 31): rtl=0.53125, hls4ml=0.00000, diff=0.53125
idx=409 (channel 1, position 51): rtl=0.50000, hls4ml=0.00000, diff=0.50000
idx=265 (channel 1, position 33): rtl=0.46875, hls4ml=0.00000, diff=0.46875
idx=325 (channel 5, position 40): rtl=0.00000, hls4ml=0.46875, diff=0.46875
idx=323 (channel 3, position 40): rtl=0.46875, hls4ml=0.00000, diff=0.46875
idx=326 (channel 6, position 40): rtl=0.43750, hls4ml=0.00000, diff=0.43750
idx=395 (channel 3, position 49): rtl=0.37500, hls4ml=0.00000, diff=0.37500
idx=413 (channel 5, position 51): rtl=0.53125, hls4ml=0.18750, diff=0.34375
idx=396 (channel 4, position 49): rtl=0.34375, hls4ml=0.00000, diff=0.34375
channel 0: 3 outliers, max diff=0.15625, mean diff=0.14583
channel 1: 10 outliers, max diff=0.53125, mean diff=0.30938
channel 3: 4 outliers, max diff=0.84375, mean diff=0.48438
channel 4: 3 outliers, max diff=0.34375, mean diff=0.23958
cha

In [48]:
pool0_rtl = load_rtl_layer(rtl_path(2, "pool0"), nfrac=6)  
pool0_ref = load_ref_layer(hls_path(2, "max_pooling2d"))

# conv1's positions 40-51 come from a specific range of pool0's output — 
# figure out pool0's own position/channel indexing (6 channels) and check 
# whether pool0 itself is already wrong at the relevant positions, or 
# whether pool0 is fine and the bug is specifically inside conv1's window/accumulate logic
print("pool0 rtl around relevant region:", pool0_rtl[40*6:52*6])
print("pool0 ref around relevant region:", pool0_ref[40*6:52*6])

pool0 rtl around relevant region: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
pool0 ref around relevant region: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
